> 在线讲义：[第 34 章 · 组合并行与模型规模：坐标、复用与最忙的那一组](https://wikieden.github.io/ai-infra-from-constraints/lessons/34-combined-parallel.html) · 本书《从约束推导 AI Infra》

# 第 34 章 · 组合并行与模型规模：坐标、复用与最忙的那一组

配套讲义：仓库里 `ai-infra/notes/lessons/34-combined-parallel.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



TP2×EP4 的归约发送量、批内专家复用、式 (6-8) 的期望活跃专家数取自 AII 第 6 章 §6.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
EXPERT = 3 * 4096 * 1536 * 2  # 36 MiB

**函数 `ring_send_factor`**：环形 AllReduce 每卡发送量是完整向量的 2(n-1)/n 倍。

In [ ]:
def ring_send_factor(n: int) -> float:
    """环形 AllReduce 每卡发送量是完整向量的 2(n-1)/n 倍。"""
    return 2 * (n - 1) / n

**函数 `tp2ep4_send_kib`**：每层三次汇合（注意力组内、专家组内、四组之间）每卡发送 KiB。

In [ ]:
def tp2ep4_send_kib(hidden: int = 4096, b: int = 4) -> float:
    """每层三次汇合（注意力组内、专家组内、四组之间）每卡发送 KiB。"""
    v = hidden * b / KiB
    return v * ring_send_factor(2) + v * ring_send_factor(2) + v * ring_send_factor(4)

**函数 `expected_active`**：式 (6-8)：m 个 token 各选 k 个，期望活跃专家数。

In [ ]:
def expected_active(E: int, k: int, m: int) -> float:
    """式 (6-8)：m 个 token 各选 k 个，期望活跃专家数。"""
    return E * (1 - (1 - k / E) ** m)

**函数 `group_load`**：assign[e] = 专家 e 收到的 token 数；返回各 EP 组的任务数。

In [ ]:
def group_load(assign: list[int], groups: int = 4, per_group: int = 32) -> list[int]:
    """assign[e] = 专家 e 收到的 token 数；返回各 EP 组的任务数。"""
    load = [0] * groups
    for e, n in enumerate(assign):
        load[e // per_group] += n
    return load

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：TP2×EP4，FP32 归约")
s = tp2ep4_send_kib()
print(f"  每 token 每卡 {s:.0f} KiB（16 + 16 + 24），八卡 {8 * s:.0f} KiB；8192 token 每卡 {s * 8192 / 1024:.0f} MiB")

In [ ]:
print("\n第二步：64 个 token × 8 个专家 = 512 次分派")
uniform = [4] * 128
hot = [64 if e < 8 else 0 for e in range(128)]
spread = [64 if e in (0, 1, 32, 33, 64, 65, 96, 97) else 0 for e in range(128)]
for name, a in (("均匀", uniform), ("集中在组 0", hot), ("热点分散到四组", spread)):
    act = sum(1 for n in a if n)
    print(f"  {name}：读 {act * EXPERT / MiB:,.0f} MiB，各组任务 {group_load(a)}")
for m in (1, 8, 64, 512):
    ea = expected_active(128, 8, m)
    print(f"  随机路由 m = {m:3d}：期望活跃 {ea:5.1f} 个，读 {ea * EXPERT / GiB:.2f} GiB")

## 3. 代码里才看得到的

1. 式 (6-8) 在 $m = 8$ 时是 51.6，原文写"约 52"。
2. `group_load` 假设专家 $e$ 在第 $e // 32$ 组：换一种放置（比如交错放）就能把集中的热点自动分散。
3. 读取量只算被选中的专家、每个读一次；如果 kernel 对每个 token 单独读专家权重，就没有批内复用。

## 4. 自己改一改

- 给 `group_load` 加一个"交错放置"（专家 $e$ 放在组 $e \bmod 4$），再跑集中的那种分派。
- 用随机数模拟 64 个 token 的路由 1000 次，画出最忙组任务数的分布。
- 把 `tp2ep4_send_kib` 改成 BF16 归约，发送量变多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 工号写部门和小组，就知道和谁开会；
> 小组先对账，部门再对账：一倍、一倍、一倍半；
> 借同几本书取得少，都在一个柜台排长队；分到四个柜台两头都好；
> 家具多了加房间，太大就拆，搬前先压缩。